In [ ]:
import os
from openai import OpenAI

# OPENAI_API_KEY is read from the environment (see .env.example)


In [ ]:
from openai import OpenAI

client = OpenAI()

train_id = client.files.create(
    file = open("../../data/legacy/training_data.jsonl", "rb"),
    purpose = "fine-tune"
)

val_id = client.files.create(
    file = open("../../data/legacy/validation_data.jsonl", "rb"),
    purpose = "fine-tune"
)

print(train_id)
print(val_id)

In [ ]:
response = client.fine_tuning.jobs.create(
    training_file = train_id.id,
    validation_file = val_id.id,
    model = "gpt-3.5-turbo",
    suffix = "HistoryGPT",
    hyperparameters = {
        "n_epochs": 5,
        "batch_size": 7,
        "learning_rate_multiplier": 0.2
    }
)

finetune_id = response.id
finetune_status = response.status

In [ ]:
print(f'Fine-tunning model with jobID: {finetune_id}.')
print(f"Training Response: {response}")
print(f"Training Status: {finetune_status}")

In [ ]:
import signal
import datetime

def stats(singal, frame):
    #Capturing the status of fine tuning from the fine tuning event ibject
    status = client.fine_tuning.jobs.retrieve(finetune_id).status
    print("Job is ", status)
    return

print("Fine Tuning Events: ")

# signal.signal(signal.SIGINT, stats)

#Capturing the event information from the fine tuning event object
tasks = client.fine_tuning.jobs.list()

print(tasks)

# try:
#     for task in tasks:
#         print(
#             f'{datetime.datetime.fromtimestamp(event.created_at)} {event.message}'
#         )
# except Exception:
#     print("Stream interrupted (client disconnected).")

In [ ]:
#Result of the fine tuning operation

import time

status = client.fine_tuning.jobs.retrieve(finetune_id).status

if status not in ["succeeded", "failed"]:
    print(f"Job not in terminal status: {finetune_status}. Waiting.")
    while status not in ["succeeded", "failed"]:
        time.sleep(2)
        status = client.fine_tuning.jobs.retrieve(finetune_id).status
        print(f"Status: {finetune_status}")
else:
    print(f"Finetune job {finetune_id} finished with status: {finetune_status}")
    
print("Checking other finetune jobs in the subscription.")
result = client.fine_tuning.jobs.list()
print(f"Found {len(result.data)} finetune jobs.")

In [ ]:
# Retrieve the finetuned model

fine_tuned_model = result.data[0].fine_tuned_model
print(fine_tuned_model)

In [ ]:
#Selecting the new prompt

new_prompt = "Generate a text talking about devoting one's life to god and to the service of others, emphasizing on the need for selflessness which takes its foundation in the Christian beliefs, as if it was written in the 15th century"

completion = client.chat.completions.create(
  model=fine_tuned_model,
  messages=[
    {"role": "system", "content": "HistoryGPT is a chatbot that generates linguistically accurate historical text from the past 500 years."},
    {"role": "user", "content": new_prompt}
  ]
)
print(completion.choices[0].message)

In [ ]:
#Retrieving the mertics file

finetune_file = client.fine_tuning.jobs.retrieve(finetune_id)
print(finetune_file.result_files[0])

content = client.files.content(finetune_file.result_files[0])

#This function returns a CSV file to the corresponding OpenAI developer account called step_metrics.csv

In [ ]:
client.files.list()

In [ ]:
client.files.retrieve('file-K2lwMgmsywU7Hhi7wqA5xfDe')

In [ ]:
import pandas as pd

df_metrics = pd.read_csv('step_metrics.csv')
print(df_metrics)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ypoints = np.array(df_metrics.train_accuracy)
xpoints = np.array(df_metrics.step)

plt.ylabel("Training Accuracy")
plt.xlabel("Step")
plt.plot(xpoints, ypoints)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ypoints = np.array(df_metrics.valid_mean_token_accuracy)
xpoints = np.array(df_metrics.step)

plt.ylabel("Validation Accuracy")
plt.xlabel("Step")
plt.plot(xpoints, ypoints)
plt.show()